# 18.2 贝叶斯逻辑回归 / Bayesian Logistic Regression (Laplace Approximation)

**中文**：上一节贝叶斯线性回归美在**共轭**——高斯先验遇高斯似然,后验还是高斯,有闭式解。但现实中**绝大多数模型都没有共轭性**。最典型的就是**逻辑回归**:sigmoid 似然遇上高斯先验,后验**不再是任何已知分布,没有闭式解**。本节学最简单、最经典的近似方法——**拉普拉斯近似(Laplace Approximation)**:*"既然真后验形状复杂,就用一个高斯去贴合它。"*
**English**: Last section's Bayesian linear regression was beautiful because of **conjugacy** — Gaussian prior meets Gaussian likelihood, posterior stays Gaussian, closed form. But in reality **most models are not conjugate**. The classic example is **logistic regression**: a sigmoid likelihood with a Gaussian prior gives a posterior that is **no longer any known distribution — no closed form**. Here we learn the simplest, most classic approximation — the **Laplace approximation**: *"since the true posterior has a complex shape, fit a Gaussian to it."*

---

**中文**：拉普拉斯近似的思路极其简单,两步:
**English**: The Laplace approximation is remarkably simple, two steps:
1. **找众数(mode)**:找到后验的最高点,即 **MAP 估计** $\mathbf w_{\text{MAP}}=\arg\max_\mathbf w\ p(\mathbf w|\mathcal D)$。这一步就是带 L2 正则的普通逻辑回归(优化)。
   **Find the mode**: locate the posterior's peak, the **MAP estimate** $\mathbf w_{\text{MAP}}=\arg\max_\mathbf w\ p(\mathbf w|\mathcal D)$. This is just L2-regularized logistic regression (optimization).
2. **用曲率拟合高斯**:在众数处对**负对数后验**求二阶导(**海森矩阵 Hessian** $H$),用它当高斯的精度。众数处越"尖"(曲率大),不确定性越小。
   **Fit a Gaussian via curvature**: at the mode, take the second derivative (**Hessian** $H$) of the **negative log posterior** and use it as the Gaussian's precision. A sharper peak (larger curvature) means less uncertainty.

$$p(\mathbf w|\mathcal D)\approx\mathcal N\big(\mathbf w_{\text{MAP}},\ \mathbf S_N\big),\qquad \mathbf S_N=H^{-1},\quad H=\underbrace{\Phi^\top \text{diag}\big(p_i(1-p_i)\big)\Phi}_{\text{似然的曲率}}+\underbrace{\alpha I}_{\text{先验}}$$

**中文**：直觉:**在众数附近，任何光滑的后验都长得像一个高斯**(泰勒二阶展开)。所以我们只要抓住"峰在哪(MAP)"和"峰有多尖(Hessian)",就用一个高斯把真后验局部贴合了。有了这个近似高斯后验，就能像上一节一样算**预测分布**(带不确定性)。
**English**: Intuition: **near its mode, any smooth posterior looks like a Gaussian** (second-order Taylor expansion). So capturing just "where the peak is (MAP)" and "how sharp it is (Hessian)" locally fits the true posterior with a Gaussian. With this approximate Gaussian posterior we can compute a **predictive distribution** (with uncertainty), like last section.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 近似推断入门必考）**
> **中文**：**Laplace 近似=用高斯贴合后验众数**:①求 MAP(=带L2的点估计);②在众数处的负对数后验 Hessian 的逆当协方差。逻辑回归无共轭→后验无闭式→Laplace 是最简近似。**预测**要对后验积分, 用 **probit 近似**:$p(y{=}1|x_*)\approx\sigma(\kappa\,\mu_a)$, $\kappa{=}1/\sqrt{1+\pi\sigma_a^2/8}$——远离数据时 $\sigma_a^2$ 大→$\kappa$ 小→预测概率**被拉向 0.5(不那么自信)**, 缓解 MLE 的**过度自信**。**局限**:只是**局部**近似(只看众数处曲率), 后验多峰/偏斜时会失真。**用途**:贝叶斯神经网络的轻量不确定性(Laplace-BNN)、模型证据近似。
> **English**: **Laplace = fit a Gaussian to the posterior mode**: ① find the MAP (= L2-regularized point estimate); ② covariance = inverse Hessian of the negative log posterior at the mode. Logistic regression is non-conjugate → no closed-form posterior → Laplace is the simplest approximation. **Prediction** integrates over the posterior via the **probit approximation**: $p(y{=}1|x_*)\approx\sigma(\kappa\,\mu_a)$, $\kappa{=}1/\sqrt{1+\pi\sigma_a^2/8}$ — far from data $\sigma_a^2$ is large → $\kappa$ small → predicted probability **pulled toward 0.5 (less confident)**, mitigating MLE's **overconfidence**. **Limits**: only a **local** approximation (curvature at the mode), distorted for multimodal/skewed posteriors. **Uses**: lightweight uncertainty for Bayesian neural nets (Laplace-BNN), model-evidence approximation.


In [ ]:

# ============================================================
# 数据:二维两类分类 / 2D two-class classification
# 中文:两团高斯点(标签0/1)。特征加上偏置项。用少量数据以便看出"远离数据处的不确定性"。
# English: two Gaussian blobs (labels 0/1). Features plus a bias. Few points so "uncertainty far from data" shows.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
np.random.seed(0)
n=25
X0=np.random.randn(n,2)*0.7+[-1.6,0.3]; X1=np.random.randn(n,2)*0.7+[1.6,-0.3]
X=np.vstack([X0,X1]); y=np.r_[np.zeros(n),np.ones(n)]
Phi=np.c_[np.ones(len(X)),X]                                  # 设计矩阵 [1, x1, x2] / design matrix
alpha=1.0                                                     # 高斯先验精度 / prior precision
def sigmoid(z): return 1/(1+np.exp(-np.clip(z,-500,500)))
print("样本数 / #samples:", len(X), "| 特征维(含偏置)/ features(with bias):", Phi.shape[1])


**中文**：第一步——**求 MAP**。用**牛顿法(等价于 IRLS)** 最小化负对数后验(= 负对数似然 + L2 先验项)。每步用梯度 $g$ 和海森 $H$ 更新 $\mathbf w\leftarrow\mathbf w-H^{-1}g$。收敛后 $\mathbf w_{\text{MAP}}$ 就是后验众数。
**English**: Step one — **find the MAP**. Use **Newton's method (equivalent to IRLS)** to minimize the negative log posterior (= negative log-likelihood + L2 prior). Each step updates $\mathbf w\leftarrow\mathbf w-H^{-1}g$ with gradient $g$ and Hessian $H$. At convergence $\mathbf w_{\text{MAP}}$ is the posterior mode.


In [ ]:

# ============================================================
# 第1步:牛顿法求 MAP / Newton's method for the MAP
# ============================================================
w=np.zeros(Phi.shape[1])
for it in range(50):
    p=sigmoid(Phi@w); R=p*(1-p)                              # 预测概率 & 方差权重 / probs & variance weights
    grad=Phi.T@(p-y) + alpha*w                               # 梯度:似然项 + 先验项 / gradient
    H=Phi.T@(R[:,None]*Phi) + alpha*np.eye(Phi.shape[1])     # 海森矩阵 / Hessian
    step=np.linalg.solve(H,grad); w=w-step                   # 牛顿更新 / Newton update
    if np.abs(step).max()<1e-8: break
w_map=w
print(f"MAP 权重 / MAP weights: {np.round(w_map,3)}  (牛顿法 {it+1} 步收敛 / converged in {it+1} steps)")


**中文**：第二步——**拉普拉斯近似的协方差**就是众数处海森矩阵的逆 $\mathbf S_N=H^{-1}$。于是近似后验 $=\mathcal N(\mathbf w_{\text{MAP}},\mathbf S_N)$。我们从这个高斯里采样若干组权重,每组对应一条决策边界——直观看到"边界的不确定性"。
**English**: Step two — the **Laplace covariance** is the inverse Hessian at the mode $\mathbf S_N=H^{-1}$. So the approximate posterior is $\mathcal N(\mathbf w_{\text{MAP}},\mathbf S_N)$. We sample several weight sets from this Gaussian, each giving a decision boundary — visualizing "boundary uncertainty."


In [ ]:

# ============================================================
# 第2步:后验协方差 + 采样决策边界 / posterior covariance + sampled boundaries
# ============================================================
p=sigmoid(Phi@w_map); R=p*(1-p)
H=Phi.T@(R[:,None]*Phi) + alpha*np.eye(Phi.shape[1])
SN=np.linalg.inv(H)                                          # 拉普拉斯近似后验协方差 / Laplace covariance
print("后验标准差(权重不确定性)/ posterior std:", np.round(np.sqrt(np.diag(SN)),3))

# 预测分布:probit 近似(对后验积分的近似)/ predictive via probit approximation
def predict_grid(XY, bayesian=True):
    P=np.c_[np.ones(len(XY)), XY]
    mu=P@w_map                                              # 激活均值 / mean activation
    if not bayesian: return sigmoid(mu)                     # MAP 点预测(过度自信)/ MAP point prediction
    var=np.sum(P@SN*P, axis=1)                              # 激活方差 / activation variance
    kappa=1/np.sqrt(1+np.pi*var/8)                          # probit 修正系数 / moderation factor
    return sigmoid(kappa*mu)                                # 被不确定性"软化"的预测 / moderated prediction

rng=np.random.default_rng(0)
w_samples=rng.multivariate_normal(w_map, SN, size=15)       # 从后验采样权重 / sample weights
print("采样了 15 组权重, 每组一条决策边界 / sampled 15 weight sets -> 15 boundaries")


In [ ]:

# ============================================================
# 可视化:MAP 点预测 vs 贝叶斯(Laplace)预测 / MAP vs Bayesian predictive
# ============================================================
gx,gy=np.meshgrid(np.linspace(-6,6,150), np.linspace(-5,5,150))
grid=np.c_[gx.ravel(),gy.ravel()]
P_map=predict_grid(grid,bayesian=False).reshape(gx.shape)
P_bay=predict_grid(grid,bayesian=True ).reshape(gx.shape)
fig,ax=plt.subplots(1,3,figsize=(17,4.8))
def scatter(a):
    a.scatter(X0[:,0],X0[:,1],c="#4C72B0",s=25,edgecolor="w"); a.scatter(X1[:,0],X1[:,1],c="#C44E52",s=25,edgecolor="w")
# ① MAP 点预测:等高线又直又密(处处自信)/ MAP: sharp parallel contours everywhere
c0=ax[0].contourf(gx,gy,P_map,levels=np.linspace(0,1,11),cmap="RdBu_r",alpha=0.8); scatter(ax[0])
ax[0].set_title("MLE/MAP 点预测:处处过度自信 / overconfident everywhere"); plt.colorbar(c0,ax=ax[0],fraction=0.046)
# ② 贝叶斯预测:远离数据处等高线散开(退向0.5)/ Bayesian: contours fan out far from data
c1=ax[1].contourf(gx,gy,P_bay,levels=np.linspace(0,1,11),cmap="RdBu_r",alpha=0.8); scatter(ax[1])
ax[1].set_title("贝叶斯(Laplace):远离数据退向0.5 / moderated far from data"); plt.colorbar(c1,ax=ax[1],fraction=0.046)
# ③ 采样的决策边界(边界的不确定性)/ sampled decision boundaries
scatter(ax[2])
for w in w_samples:
    if abs(w[2])>1e-3:
        yb=-(w[0]+w[1]*np.array([-6,6]))/w[2]; ax[2].plot([-6,6],yb,"gray",alpha=0.4)
yb=-(w_map[0]+w_map[1]*np.array([-6,6]))/w_map[2]; ax[2].plot([-6,6],yb,"k",lw=2,label="MAP 边界")
ax[2].set_xlim(-6,6); ax[2].set_ylim(-5,5); ax[2].set_title("从后验采样的决策边界 / sampled boundaries"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/bay02_viz.png",dpi=80); plt.show()
# 量化远离数据处的"软化" / quantify moderation far from data
far=np.array([[5.0,4.0]])
print(f"远角点(5,4): MAP 预测 {predict_grid(far,False)[0]:.3f}  贝叶斯 {predict_grid(far,True)[0]:.3f} (更接近0.5=更谦虚)")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **贝叶斯逻辑回归缓解了"过度自信"**:对比左右两图——MLE/MAP 点预测(左)的概率等高线**又直又密、延伸到无穷远都一样自信**,哪怕在离任何数据都很远的角落也敢说"99.9% 是红类"。贝叶斯预测(中)则不同:**离数据越远,等高线越散开、预测概率越退向 0.5**(=坦承"这里我没见过数据,不确定")。这就是拉普拉斯近似的价值——**把过度自信的点估计校准得更谦虚**,在安全攸关的场景极重要。
2. **决策边界本身也有分布**:右图从后验采样的多条边界,在数据密集处几乎重合(确定)、向两侧张开(不确定)。点估计只给你一条"唯一正确"的边界,贝叶斯告诉你"边界大概在这一带"。
3. **诚实的局限**:拉普拉斯近似**只在众数附近用二阶泰勒展开贴一个高斯**——如果真后验是**多峰**(多个解)或**强烈偏斜**的,这个局部高斯就贴不准(它会漏掉其他峰、也假设对称)。所以它是"最省事但最粗糙"的近似;更准的要用 **MCMC(精确采样,18.3)** 或 **变分推断(全局优化一个分布族,18.4)**。probit 近似本身也是对"高斯后验下的 sigmoid 积分"的近似。

**English**:
1. **Bayesian logistic mitigates overconfidence**: compare left vs middle — the MLE/MAP point prediction's probability contours (left) are **straight, dense, and equally confident out to infinity**, happily claiming "99.9% red" even in corners far from any data. The Bayesian prediction (middle) differs: **the farther from data, the more contours fan out and probabilities retreat toward 0.5** (admitting "I've seen no data here, I'm unsure"). This is Laplace's value — **calibrating overconfident point estimates to be humbler**, crucial in safety-critical settings.
2. **The decision boundary itself has a distribution**: the sampled boundaries (right) nearly coincide where data is dense (certain) and fan out to the sides (uncertain). A point estimate gives one "uniquely correct" boundary; Bayesian says "the boundary is somewhere in this band."
3. **Honest limits**: Laplace fits a Gaussian via a second-order Taylor expansion **only near the mode** — if the true posterior is **multimodal** (multiple solutions) or **strongly skewed**, this local Gaussian is inaccurate (it misses other modes and assumes symmetry). So it is the "cheapest but crudest" approximation; more accurate methods are **MCMC (exact sampling, 18.3)** or **variational inference (globally optimize over a distribution family, 18.4)**. The probit approximation itself also approximates the sigmoid integral under a Gaussian posterior.

> 💼 **实战视角 / Practical angle**
> **中文**:Laplace 近似的实战用途:①给已训练好的模型(逻辑回归、甚至**神经网络** Laplace-BNN)**低成本地加不确定性**——只需在最优点算一次 Hessian,不用重训;②模型证据/边际似然近似(做模型选择);③**校准(calibration)**——深度模型常过度自信,不确定性估计能改善。**权衡**:①只抓一个峰,多峰会漏;②高维 Hessian 求逆贵(常用对角/KFAC 近似);③假设后验对称。面试金句:*"Laplace 近似=在 MAP 处用负对数后验的 Hessian 逆当协方差, 拿高斯贴合后验; 它给逻辑回归/BNN 低成本的不确定性, 但只是局部近似, 多峰失效——那时要 MCMC 或变分。"*
> **English**: Practical uses of Laplace: ① add **cheap uncertainty** to an already-trained model (logistic regression, even **neural nets** Laplace-BNN) — just one Hessian at the optimum, no retraining; ② approximate model evidence / marginal likelihood (for model selection); ③ **calibration** — deep models are often overconfident, and uncertainty estimates help. **Trade-offs**: ① captures only one mode, misses multimodality; ② high-dim Hessian inversion is costly (use diagonal/KFAC approximations); ③ assumes a symmetric posterior. Interview line: *"Laplace fits a Gaussian to the posterior using the inverse Hessian of the negative log posterior at the MAP; it gives cheap uncertainty for logistic regression / BNNs, but is only local and fails for multimodal posteriors — then use MCMC or variational inference."*

---
### 小结 / Summary
- **中文**:逻辑回归无共轭→后验无闭式; Laplace 近似=在 MAP 众数处用 Hessian 逆当协方差拟合高斯。
- **English**: Logistic regression is non-conjugate → no closed-form posterior; Laplace fits a Gaussian at the MAP mode using the inverse Hessian as covariance.
- **中文**:预测用 probit 近似, 远离数据处预测退向 0.5——缓解点估计的过度自信。
- **English**: Prediction uses the probit approximation; far from data predictions retreat toward 0.5 — mitigating point estimates' overconfidence.
- **中文**:只是局部近似(单峰、对称假设); 多峰/复杂后验要用 MCMC 或变分推断(后续)。
- **English**: Only a local approximation (single mode, symmetric); multimodal/complex posteriors need MCMC or variational inference (coming up).
